In [1]:
import torch
import clip
from PIL import Image
import numpy as np
import os
from typing import List, Tuple

In [2]:
def load_images(image_dir: str) -> Tuple[List[Image.Image], List[str]]:
    """Load all images from the specified directory."""
    images = []
    image_paths = []
    
    valid_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.webp']
    
    for filename in sorted(os.listdir(image_dir)):
        if any(filename.lower().endswith(ext) for ext in valid_extensions):
            path = os.path.join(image_dir, filename)
            image = Image.open(path).convert('RGB')
            images.append(image)
            image_paths.append(filename)
    
    return images, image_paths

def compute_richdreamer_clip_score(images: List[Image.Image], 
                                   text_prompt: str,
                                   model_name: str = "ViT-L/14",
                                   remove_outliers: bool = True) -> dict:
    """
    Compute CLIP scores following RichDreamer's evaluation protocol.
    
    Args:
        images: List of PIL images (different views of the object)
        text_prompt: Text prompt to evaluate against
        model_name: CLIP model to use (RichDreamer uses ViT-G/14, but it's not in the standard CLIP)
        remove_outliers: Whether to remove highest and lowest scores before averaging
    
    Returns:
        Dictionary with individual scores and average score
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # Load CLIP model
    print(f"Loading CLIP model: {model_name}")
    model, preprocess = clip.load(model_name, device=device)
    
    # Get the temperature (logit_scale) parameter
    logit_scale = model.logit_scale.exp().item()
    print(f"Model temperature (exp(logit_scale)): {logit_scale:.2f}")
    
    # Preprocess images
    image_tensors = torch.stack([preprocess(img) for img in images]).to(device)
    
    # Encode text
    text = clip.tokenize([text_prompt]).to(device)
    
    # Compute features
    with torch.no_grad():
        image_features = model.encode_image(image_tensors)
        text_features = model.encode_text(text)
        
        # Normalize features
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        
        # Compute similarity scores with temperature scaling
        # This is the key difference - we multiply by the learned temperature
        similarities = (image_features @ text_features.T).squeeze()
        clip_scores = (similarities * logit_scale).cpu().numpy()
    
    # Process scores
    individual_scores = clip_scores.tolist()
    
    if remove_outliers and len(individual_scores) > 2:
        # Remove highest and lowest scores as per RichDreamer
        sorted_scores = sorted(individual_scores)
        filtered_scores = sorted_scores[1:-1]
        average_score = np.mean(filtered_scores)
        print(f"Removed outliers: {sorted_scores[0]:.2f} (min) and {sorted_scores[-1]:.2f} (max)")
    else:
        filtered_scores = individual_scores
        average_score = np.mean(individual_scores)
    
    return {
        'individual_scores': individual_scores,
        'filtered_scores': filtered_scores,
        'average_score': average_score,
        'logit_scale': logit_scale
    }

def evaluate_uniform_albedo_geometry(image_dir: str, text_prompt: str):
    """
    Evaluate geometry with uniform albedo following RichDreamer's protocol.
    """
    print("=== RichDreamer-style CLIP Evaluation ===")
    print(f"Text prompt: '{text_prompt}'")
    print(f"Image directory: {image_dir}")
    print()
    
    # Load images
    images, image_names = load_images(image_dir)
    print(f"Loaded {len(images)} views: {image_names}")
    print()
    
    # Note: RichDreamer uses ViT-G/14, but it's not available in the standard CLIP
    # We'll use ViT-L/14 as the closest alternative
    # You might need to use open_clip for ViT-G/14
    results = compute_richdreamer_clip_score(
        images, 
        text_prompt,
        model_name="ViT-L/14",  # Use ViT-L/14 as ViT-G/14 isn't in standard CLIP
        remove_outliers=True
    )
    
    print("\n=== Results ===")
    print("Individual scores for each view:")
    for i, (name, score) in enumerate(zip(image_names, results['individual_scores'])):
        print(f"  {name}: {score:.2f}")
    
    print(f"\nFiltered scores (after removing outliers): {[f'{s:.2f}' for s in results['filtered_scores']]}")
    print(f"Average CLIP Score: {results['average_score']:.2f}")
    
    print("\n=== Interpretation ===")
    print(f"The scores are scaled by the model's learned temperature ({results['logit_scale']:.2f})")
    print("This is why they're in the ~20s range rather than 0-1")
    print("\nNote: RichDreamer uses ViT-G/14 which may have a different temperature than ViT-L/14")
    
    # Also show normalized scores for reference
    normalized_avg = results['average_score'] / results['logit_scale']
    print(f"\nFor reference, normalized score (0-1 range): {normalized_avg:.4f}")

In [3]:
def main():
    # Example usage - adjust these parameters
    image_dir = "test_images"  # Your directory with rendered views
    text_prompt = "super mario"  # The text prompt used for generation
    
    # Run evaluation
    evaluate_uniform_albedo_geometry(image_dir, text_prompt)
    
    print("\n" + "="*60)
    print("To exactly replicate RichDreamer's results, you would need:")
    print("1. Render your geometry with uniform albedo (no texture)")
    print("2. Generate exactly 16 views of each object")
    print("3. Use ViT-G/14 model (available via open_clip)")
    print("4. Remove highest and lowest scores before averaging")



In [4]:
main()

=== RichDreamer-style CLIP Evaluation ===
Text prompt: 'super mario'
Image directory: test_images

Loaded 4 views: ['mario_back.jpeg', 'mario_front.jpeg', 'mario_left.jpeg', 'mario_right.jpeg']

Loading CLIP model: ViT-L/14


c:\Users\ahmmo\anaconda3\envs\cfe\lib\site-packages\clip\clip.py:57: UserWarning: C:\Users\ahmmo/.cache/clip\ViT-L-14.pt exists, but the SHA256 checksum does not match; re-downloading the file
  warnings.warn(f"{download_target} exists, but the SHA256 checksum does not match; re-downloading the file")
  4%|█▌                                    | 36.5M/890M [00:10<04:07, 3.62MiB/s]


KeyboardInterrupt: 